# Matched Filter Pipeline Demo (EnMAP)

This notebook runs the CH$\_4$ detection pipeline on an EnMAP L1B scene to illustrate how the matched filter, LUT-derived targets, and propagated noise cubes generate concentration/uncertainty rasters.

**Pipeline stages**
1. Bootstrap the repository context so the helper modules in `scripts/` can be imported from the notebook.
2. Declare the scene assets, spectral window, and k-means configuration that parameterise the matched filter.
3. Execute `enmap_pipeline.ch4_detection_enmap`, which handles radiance calibration, background statistics, LUT target synthesis, and the matched-filter solve $w = \Sigma^{-1}(t-\mu)$ followed by $\Delta X = w^\top (L-\mu)$.
4. Load the RGB, concentration, and $\sigma_{RMN}$ rasters to interpret plume structure versus instrument noise.


## 1. Bootstrap repository context
Add the repository root to `PYTHONPATH`, locate the `scripts/` directory, and emit the resolved paths. This guarantees that the remainder of the notebook reuses the exact same pipeline code as the CLI, avoiding environment drift between experiments and notebook runs.


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from osgeo import gdal

NOTEBOOK_ROOT = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_ROOT
while not (REPO_ROOT / 'scripts').exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'scripts').exists():
    raise RuntimeError('Could not locate repository root containing scripts directory.')

os.environ.setdefault('PYTHONPATH', str(REPO_ROOT))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f'Notebook root: {NOTEBOOK_ROOT}')
print(f'Repository root: {REPO_ROOT}')

## 2. Scene configuration and assumptions
The configuration dictionary pins the VNIR/SWIR GeoTIFFs, metadata XML, LUT file, and precomputed SNR reference. `min_wavelength`/`max_wavelength` restrict the matched filter to absorption features (here 2.1-2.45 um), while `k` controls how many k-means clusters are used to model background covariance. The LUT mixes simulated spectra at increasing columns (0-64,000 ppm·m) so the pipeline can interpolate a target signature that matches the scene's solar zenith angle, water vapour, and elevation.


In [ ]:
SENSOR = 'enmap'
PIPELINE_OUTPUT_ROOT = NOTEBOOK_ROOT / 'outputs' / 'pipeline_demo' / SENSOR
SELECTED_MF_MODE = 'full-column'  # Options: 'srf-column', 'full-column', 'advanced', 'jpl'

ENMAP_SCENES = {
    'turkmenistan_plume_enmap': {
        'vnir': Path('/mnt/d/Lavoro/Assegno_Ricerca_Sapienza/CLEAR_UP/CH4_detection/Matched_filter_approach/hygas/test_data/enmap/Turkmenistan_20221002/20221002T074833/L1B_20221002T074833Z_002_V010501_20241110T222710Z/ENMAP01-____L1B-DT0000004147_20221002T074833Z_002_V010501_20241110T222710Z-SPECTRAL_IMAGE_VNIR.TIF'),
        'swir': Path('/mnt/d/Lavoro/Assegno_Ricerca_Sapienza/CLEAR_UP/CH4_detection/Matched_filter_approach/hygas/test_data/enmap/Turkmenistan_20221002/20221002T074833/L1B_20221002T074833Z_002_V010501_20241110T222710Z/ENMAP01-____L1B-DT0000004147_20221002T074833Z_002_V010501_20241110T222710Z-SPECTRAL_IMAGE_SWIR.TIF'),
        'metadata': Path('/mnt/d/Lavoro/Assegno_Ricerca_Sapienza/CLEAR_UP/CH4_detection/Matched_filter_approach/hygas/test_data/enmap/Turkmenistan_20221002/20221002T074833/L1B_20221002T074833Z_002_V010501_20241110T222710Z/ENMAP01-____L1B-DT0000004147_20221002T074833Z_002_V010501_20241110T222710Z-METADATA.XML'),
        'lut': Path('/mnt/d/Lavoro/Assegno_Ricerca_Sapienza/CLEAR_UP/CH4_detection/Matched_filter_approach/LUTs/CH4_lut.hdf5'),
        'snr_reference': REPO_ROOT / 'notebooks' / 'outputs' / 'enmap' / 'L1B-DT0000001584_20220712T104302Z_001_V010502_20251017T093724Z' / 'snr_reference_columnwise.npz',
        'output_dir': PIPELINE_OUTPUT_ROOT / 'turkmenistan_plume',
        'min_wavelength': 2100.0,
        'max_wavelength': 2450.0,
        'k': 1,
        'mf_mode': SELECTED_MF_MODE,
    },
    'agadez_background_enmap': {
        'vnir': Path('/mnt/d/Lavoro/Assegno_Ricerca_Sapienza/CLEAR_UP/CH4_detection/Matched_filter_approach/hygas/test_data/enmap/Agadez_Niger_20220712/L1B-DT0000001584_20220712T104302Z_001_V010502_20251017T093724Z/ENMAP01-____L1B-DT0000001584_20220712T104302Z_001_V010502_20251017T093724Z-SPECTRAL_IMAGE_VNIR.TIF'),
        'swir': Path('/mnt/d/Lavoro/Assegno_Ricerca_Sapienza/CLEAR_UP/CH4_detection/Matched_filter_approach/hygas/test_data/enmap/Agadez_Niger_20220712/L1B-DT0000001584_20220712T104302Z_001_V010502_20251017T093724Z/ENMAP01-____L1B-DT0000001584_20220712T104302Z_001_V010502_20251017T093724Z-SPECTRAL_IMAGE_SWIR.TIF'),
        'metadata': Path('/mnt/d/Lavoro/Assegno_Ricerca_Sapienza/CLEAR_UP/CH4_detection/Matched_filter_approach/hygas/test_data/enmap/Agadez_Niger_20220712/L1B-DT0000001584_20220712T104302Z_001_V010502_20251017T093724Z/ENMAP01-____L1B-DT0000001584_20220712T104302Z_001_V010502_20251017T093724Z-METADATA.XML'),
        'lut': Path('/mnt/d/Lavoro/Assegno_Ricerca_Sapienza/CLEAR_UP/CH4_detection/Matched_filter_approach/LUTs/CH4_lut.hdf5'),
        'snr_reference': REPO_ROOT / 'notebooks' / 'outputs' / 'enmap' / 'L1B-DT0000001584_20220712T104302Z_001_V010502_20251017T093724Z' / 'snr_reference_columnwise.npz',
        'output_dir': PIPELINE_OUTPUT_ROOT / 'agadez_background',
        'min_wavelength': 2100.0,
        'max_wavelength': 2450.0,
        'k': 1,
        'mf_mode': SELECTED_MF_MODE,
    },
    'BuenosAires_20240112_enmap': {
        'vnir': Path('/mnt/d/Lavoro/Assegno_Ricerca_Sapienza/CLEAR_UP/CH4_detection/Matched_filter_approach/hygas/case_studies_data/BuenosAires_20240112/EnMAP/ENMAP01-____L1B-DT0000058121_20240112T144653Z_002_V010501_20241021T060427Z-SPECTRAL_IMAGE_VNIR.TIF'),
        'swir': Path('/mnt/d/Lavoro/Assegno_Ricerca_Sapienza/CLEAR_UP/CH4_detection/Matched_filter_approach/hygas/case_studies_data/BuenosAires_20240112/EnMAP/ENMAP01-____L1B-DT0000058121_20240112T144653Z_002_V010501_20241021T060427Z-SPECTRAL_IMAGE_SWIR.TIF'),
        'metadata': Path('/mnt/d/Lavoro/Assegno_Ricerca_Sapienza/CLEAR_UP/CH4_detection/Matched_filter_approach/hygas/case_studies_data/BuenosAires_20240112/EnMAP/ENMAP01-____L1B-DT0000058121_20240112T144653Z_002_V010501_20241021T060427Z-METADATA.XML'),
        'lut': Path('/mnt/d/Lavoro/Assegno_Ricerca_Sapienza/CLEAR_UP/CH4_detection/Matched_filter_approach/LUTs/CH4_lut.hdf5'),
        'snr_reference': REPO_ROOT / 'notebooks' / 'outputs' / 'enmap' / 'L1B-DT0000001584_20220712T104302Z_001_V010502_20251017T093724Z' / 'snr_reference_columnwise.npz',
        'output_dir': PIPELINE_OUTPUT_ROOT / 'BuenosAires_20240112',
        'min_wavelength': 2100.0,
        'max_wavelength': 2450.0,
        'k': 1,
        'mf_mode': SELECTED_MF_MODE,
    },
    'Turkmenistan_20240911_enmap': {
        'vnir': Path('/mnt/d/Lavoro/Assegno_Ricerca_Sapienza/CLEAR_UP/CH4_detection/Matched_filter_approach/hygas/case_studies_data/Turkmenistan_20240911/enmap/ENMAP01-____L1B-DT0000092488_20240911T075547Z_001_V010502_20241207T112410Z-SPECTRAL_IMAGE_VNIR.TIF'),
        'swir': Path('/mnt/d/Lavoro/Assegno_Ricerca_Sapienza/CLEAR_UP/CH4_detection/Matched_filter_approach/hygas/case_studies_data/Turkmenistan_20240911/enmap/ENMAP01-____L1B-DT0000092488_20240911T075547Z_001_V010502_20241207T112410Z-SPECTRAL_IMAGE_SWIR.TIF'),
        'metadata': Path('/mnt/d/Lavoro/Assegno_Ricerca_Sapienza/CLEAR_UP/CH4_detection/Matched_filter_approach/hygas/case_studies_data/Turkmenistan_20240911/enmap/ENMAP01-____L1B-DT0000092488_20240911T075547Z_001_V010502_20241207T112410Z-METADATA.XML'),
        'lut': Path('/mnt/d/Lavoro/Assegno_Ricerca_Sapienza/CLEAR_UP/CH4_detection/Matched_filter_approach/LUTs/CH4_lut.hdf5'),
        'snr_reference': REPO_ROOT / 'notebooks' / 'outputs' / 'enmap' / 'L1B-DT0000001584_20220712T104302Z_001_V010502_20251017T093724Z' / 'snr_reference_columnwise.npz',
        'output_dir': PIPELINE_OUTPUT_ROOT / 'Turkmenistan_20240911',
        'min_wavelength': 2100.0,
        'max_wavelength': 2450.0,
        'k': 1,
        'mf_mode': SELECTED_MF_MODE,
    },
    
}

SELECTED_SCENE = 'Turkmenistan_20240911_enmap'  # switch to 'agadez_background' for the methane-free calibration case
CONFIG = ENMAP_SCENES[SELECTED_SCENE]

CONFIG['output_dir'].mkdir(parents=True, exist_ok=True)
CONFIG


## 3. Run the matched-filter pipeline
`enmap_pipeline.ch4_detection_enmap` performs the operational workflow: destriping/equalisation, k-means clustering, covariance estimation, LUT-based target synthesis, and the matched filter solve. The filter weights are derived from $w = \Sigma^{-1}(t-\mu)$ and applied column-wise to radiance residuals to obtain path-integrated enhancements. The function also propagates the SNR reference to produce $\sigma_{RMN}$, assuming temporally stable detector noise and Gaussian statistics.


In [ ]:
from scripts.pipelines import enmap_pipeline

enmap_pipeline.ch4_detection_enmap(
    vnir_file=str(CONFIG['vnir']),
    swir_file=str(CONFIG['swir']),
    metadata_file=str(CONFIG['metadata']),
    lut_file=str(CONFIG['lut']),
    output_dir=str(CONFIG['output_dir']),
    k=CONFIG['k'],
    min_wavelength=CONFIG['min_wavelength'],
    max_wavelength=CONFIG['max_wavelength'],
    mf_mode=CONFIG.get('mf_mode', 'srf-column'),
    snr_reference_path=str(CONFIG['snr_reference'])
)
print('Pipeline run complete.')

## 4. Load and interpret retrieval outputs
The helper routines locate the most recent RGB/composite/uncertainty rasters inside `output_dir`, read them with GDAL, and normalise the RGB channels for display. The concentration panel shows $\Delta X$ (ppm·m) constrained to 0-8,000 to highlight plume cores, whereas the uncertainty panel visualises $\sigma_{RMN}$ (instrument noise propagated through the matched filter) in ppm·m. Comparing both maps helps distinguish signal-dominated regions (high enhancement with low $\sigma_{RMN}$) from noise-limited artefacts.


In [ ]:
from scripts.satellites import enmap_utils

def _read_raster(path: Path):
    ds = gdal.Open(str(path))
    if ds is None:
        raise RuntimeError(f'Failed to open raster: {path}')
    arr = ds.ReadAsArray().astype(np.float32)
    ds = None
    return arr

basename = enmap_utils.derive_basename_from_metadata(str(CONFIG['metadata']))
rgb_path = CONFIG['output_dir'] / f'{basename}_RGB.tif'
conc_path = CONFIG['output_dir'] / f'{basename}_MF.tif'
unc_path = CONFIG['output_dir'] / f'{basename}_MF_uncertainty.tif'

for label, path in [('RGB', rgb_path), ('Concentration', conc_path), ('Uncertainty', unc_path)]:
    if not path.exists():
        raise FileNotFoundError(f"Expected output not found: {path}")
    print(f"{label}: {path}")

rgb = _read_raster(rgb_path)
concentration = _read_raster(conc_path)
uncertainty = _read_raster(unc_path)

if rgb.ndim == 3:
    rgb_img = np.transpose(rgb, (1, 2, 0))
else:
    rgb_img = rgb

def _normalize_rgb(arr: np.ndarray) -> np.ndarray:
    high = np.quantile(arr, 0.995, axis=(0, 1), keepdims=True)
    high = np.clip(high, 1e-3, None)
    arr = np.clip(arr, 0, high)
    arr = arr / high
    return np.clip(arr, 0, 1)

rgb_disp = _normalize_rgb(rgb_img.astype(np.float32))

fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=600)
axes[0].imshow(rgb_disp)
axes[0].set_title('RGB composite')
axes[0].axis('off')

im1 = axes[1].imshow(concentration, cmap='plasma', vmin=0, vmax=8000)
axes[1].set_title('CH$_4$ enhancement (ppm·m)')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(uncertainty, cmap='plasma', vmin=120, vmax=200)
axes[2].set_title('Instrument noise σ$_{RMN}$ (ppm·m)')
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

for ax in axes[1:]:
    ax.axis('off')

fig.suptitle(SENSOR.upper() + ' matched-filter outputs', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
import numpy as np
# Import the necessary toolkit for precise colorbar placement
from mpl_toolkits.axes_grid1 import make_axes_locatable

# --- Steps 1 & 2: Grid Definition and Warping (unchanged from previous correct version) ---

# --- Step 0: Define Paths ---
# basename = enmap_utils.derive_basename_from_metadata(str(CONFIG['metadata']))
# rgb_path = CONFIG['output_dir'] / f'{basename}_RGB.tif'
# conc_path = CONFIG['output_dir'] / f'{basename}_MF.tif'
# unc_path = CONFIG['output_dir'] / f'{basename}_MF_uncertainty.tif'

# --- Step 1: Define the Common Target Grid ---
print("Defining common grid based on CH4 map...")
with rasterio.open(conc_path) as master_ds:
    dst_crs = master_ds.crs
    transform, width, height = calculate_default_transform(
        master_ds.crs, dst_crs, master_ds.width, master_ds.height, *master_ds.bounds
    )
    dst_kwargs = master_ds.meta.copy()
    dst_kwargs.update({
        'crs': dst_crs, 'transform': transform, 'width': width, 'height': height, 'nodata': 0
    })
print(f"Target grid defined: {width}x{height} pixels, north-up.")

# --- Step 2: Warp all three datasets to this common grid ---
print("Warping Concentration data...")
conc_warped = np.zeros((height, width), dtype=np.float32)
with rasterio.open(conc_path) as src:
    reproject(
        source=rasterio.band(src, 1), destination=conc_warped,
        src_transform=src.transform, src_crs=src.crs,
        dst_transform=transform, dst_crs=dst_crs, resampling=Resampling.bilinear
    )

print("Warping Uncertainty data...")
unc_warped = np.zeros((height, width), dtype=np.float32)
with rasterio.open(unc_path) as src:
    reproject(
        source=rasterio.band(src, 1), destination=unc_warped,
        src_transform=src.transform, src_crs=src.crs,
        dst_transform=transform, dst_crs=dst_crs, resampling=Resampling.bilinear
    )

print("Warping RGB data...")
rgb_warped = np.zeros((3, height, width), dtype=np.float32)
with rasterio.open(rgb_path) as src:
    for i in range(1, 4):
        reproject(
            source=rasterio.band(src, i), destination=rgb_warped[i-1],
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=transform, dst_crs=dst_crs, resampling=Resampling.bilinear
        )

# --- Step 3: Plot the aligned, warped data with precise colorbar placement ---
print("Plotting warped data...")

# Prepare data for display
rgb_disp = np.transpose(rgb_warped, (1, 2, 0))
def _normalize_rgb(arr: np.ndarray) -> np.ndarray:
    high = np.nanquantile(arr, 0.995, axis=(0, 1), keepdims=True)
    high = np.clip(high, 1e-3, None)
    arr = np.clip(arr, 0, high)
    arr = arr / high
    return np.clip(arr, 0, 1)
rgb_disp = _normalize_rgb(rgb_disp)

extent = [transform[2], transform[2] + transform[0]*width, 
          transform[5] + transform[4]*height, transform[5]]

# Create figure and axes. Using constrained_layout=True helps manage spacing.
fig, axes = plt.subplots(1, 3, figsize=(20, 7), dpi=150, constrained_layout=True)
fig.suptitle(SENSOR.upper() + ' Matched-Filter Outputs (Warped to Common North-Up Grid)', fontsize=16)

# Plot RGB
axes[0].imshow(rgb_disp, extent=extent)
axes[0].set_title('RGB Composite')

# Plot Concentration
vmin_conc, vmax_conc = 0, 8000
im1 = axes[1].imshow(conc_warped, cmap='plasma', vmin=vmin_conc, vmax=vmax_conc, extent=extent)
axes[1].set_title('CH$_4$ enhancement (ppm·m)')

# *** NEW: Create a dedicated axis for the concentration colorbar ***
divider1 = make_axes_locatable(axes[1])
cax1 = divider1.append_axes("right", size="5%", pad=0.1)
fig.colorbar(im1, cax=cax1, label='ppm·m')

# Plot Uncertainty
vmin_unc, vmax_unc = 120, 200
im2 = axes[2].imshow(unc_warped, cmap='plasma', vmin=vmin_unc, vmax=vmax_unc, extent=extent)
axes[2].set_title('Instrument noise σ$_{RMN}$ (ppm·m)')

# *** NEW: Create a dedicated axis for the uncertainty colorbar ***
divider2 = make_axes_locatable(axes[2])
cax2 = divider2.append_axes("right", size="5%", pad=0.1)
fig.colorbar(im2, cax=cax2, label='ppm·m')

# Final formatting
for ax in axes:
    ax.set_aspect('equal')
    ax.set_xlabel('Easting (m)')
    ax.tick_params(axis='x', rotation=45)
axes[0].set_ylabel('Northing (m)')
for ax in axes[1:]:
    ax.tick_params(labelleft=False)

# We use constrained_layout=True instead of tight_layout for better results with this method
# plt.tight_layout(rect=[0, 0.03, 1, 0.95]) 
plt.show()


In [ ]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
import numpy as np
from mpl_toolkits.axes_grid1 import make_axes_locatable
from pathlib import Path

# --- Step 0: Define Paths ---
# Ensure these variables from the notebook's context are available
# CONFIG, SENSOR, basename, conc_path, unc_path, rgb_path

output_dir = Path(CONFIG['output_dir'])
output_dir.mkdir(exist_ok=True)

# --- Steps 1 & 2: Grid Definition and Warping (Unchanged) ---
print("Defining common grid based on CH4 map...")
with rasterio.open(conc_path) as master_ds:
    dst_crs = master_ds.crs
    transform, width, height = calculate_default_transform(
        master_ds.crs, dst_crs, master_ds.width, master_ds.height, *master_ds.bounds
    )

print("Warping source data to the common grid...")
# (Warping for conc_warped and unc_warped is unchanged)
conc_warped = np.zeros((height, width), dtype=np.float32)
with rasterio.open(conc_path) as src:
    reproject(
        source=rasterio.band(src, 1), destination=conc_warped,
        src_transform=src.transform, src_crs=src.crs,
        dst_transform=transform, dst_crs=dst_crs, resampling=Resampling.bilinear
    )

unc_warped = np.zeros((height, width), dtype=np.float32)
with rasterio.open(unc_path) as src:
    reproject(
        source=rasterio.band(src, 1), destination=unc_warped,
        src_transform=src.transform, src_crs=src.crs,
        dst_transform=transform, dst_crs=dst_crs, resampling=Resampling.bilinear
    )

rgb_warped = np.zeros((3, height, width), dtype=np.float32)
with rasterio.open(rgb_path) as src:
    for i in range(1, 4):
        reproject(
            source=rasterio.band(src, i), destination=rgb_warped[i-1],
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=transform, dst_crs=dst_crs, resampling=Resampling.bilinear
        )

# --- Data Preparation for Plotting (with the FIX) ---

# Transpose RGB for plotting: (bands, height, width) -> (height, width, bands)
rgb_disp = np.transpose(rgb_warped, (1, 2, 0))

# *** THE FIX IS HERE ***
# Create a mask for nodata pixels (where all 3 bands are 0)
nodata_mask = np.all(rgb_disp == 0, axis=2)
# Set these pixels to NaN so they are ignored by nanquantile
rgb_disp[nodata_mask] = np.nan

# Define the robust normalization function
def _normalize_rgb(arr: np.ndarray) -> np.ndarray:
    """Normalizes an RGB array, ignoring NaN values."""
    # Use nanquantile to correctly calculate percentile on data, ignoring NaNs
    high = np.nanquantile(arr, 0.995, axis=(0, 1), keepdims=True)
    high = np.clip(high, 1e-3, None) # Prevent division by zero
    
    # Normalize the array
    arr = np.clip(arr, 0, high)
    arr = arr / high
    
    # Convert NaNs back to 0 (black) for display and clip to [0, 1] range
    return np.nan_to_num(np.clip(arr, 0, 1))

# Apply the normalization
rgb_disp = _normalize_rgb(rgb_disp)

# Get the geographic extent for imshow from the new transform
extent = [transform[2], transform[2] + transform[0]*width, 
          transform[5] + transform[4]*height, transform[5]]

# --- Step 3: Create and Save Each Plot Individually (Unchanged) ---

plot_configs = [
    {
        'data': rgb_disp,
        'title': f'{SENSOR.upper()} RGB Composite',
        'filename': output_dir / f'{basename}_RGB_georeferenced.png',
        'is_rgb': True
    },
    # ... (rest of the plot_configs list is the same)
    {
        'data': conc_warped,
        'title': f'{SENSOR.upper()} CH$_4$ Enhancement (ppm·m)',
        'filename': output_dir / f'{basename}_MF_georeferenced.png',
        'is_rgb': False,
        'cmap': 'plasma',
        'vmin': 0,
        'vmax': 8000,
        'cbar_label': 'ppm·m'
    },
    {
        'data': unc_warped,
        'title': f'{SENSOR.upper()} Instrument Noise σ$_{{RMN}}$ (ppm·m)',
        'filename': output_dir / f'{basename}_MF_uncertainty_georeferenced.png',
        'is_rgb': False,
        'cmap': 'plasma',
        'vmin': 120,
        'vmax': 200,
        'cbar_label': 'ppm·m'
    }
]

# Loop through the configurations and generate each plot
for config in plot_configs:
    print(f"Generating plot: {config['filename'].name}")
    
    fig, ax = plt.subplots(figsize=(8, 8), dpi=200, constrained_layout=True)
    
    if config['is_rgb']:
        im = ax.imshow(config['data'], extent=extent)
    else:
        # For non-RGB plots, replace 0s with NaN to avoid plotting nodata areas
        data_to_plot = config['data'].copy()
        data_to_plot[data_to_plot == 0] = np.nan
        im = ax.imshow(
            data_to_plot, 
            extent=extent, 
            cmap=config['cmap'], 
            vmin=config['vmin'], 
            vmax=config['vmax']
        )
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        fig.colorbar(im, cax=cax, label=config['cbar_label'])

    ax.set_title(config['title'])
    ax.set_aspect('equal')
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    ax.tick_params(axis='x', rotation=45)
    
    plt.savefig(config['filename'], bbox_inches='tight')
    plt.show()
    plt.close(fig)

print("\nAll individual plots have been generated and saved.")
